# 最近进展与超额误差选模型

单机制、冻结四候选、严格成熟年度验证；已复用历史的诊断实验。新候选内层拟合为0，外层拟合最多11次。

控制：原平均两腿MSE选择。候选：SOXL-minus-SOXX H5收益差MSE选择。直接SOXX按相同窗口和费用比较。Qlib DatasetH、Recorder、原生Backtest和risk_analysis是证据入口。

In [1]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
ENTRY_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "AGENTS.md").is_file())
EVIDENCE_ROOT = Path(os.environ.get("SOXL_EVIDENCE_ROOT", ENTRY_ROOT)).resolve()
SOURCE_ROOT = Path(os.environ.get("SOXL_RESEARCH_SOURCE_ROOT", ENTRY_ROOT)).resolve()
sys.path.insert(0, str(SOURCE_ROOT))
CARD = ENTRY_ROOT / "research" / "benchmark_relative_loss_selection_20261012"
PLAN = json.loads((CARD / "plan.json").read_text())
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.workflow import R
from src.workflow.relative_loss_experiment import load_frozen_selection_inputs, run_relative_loss_experiment
qlib.init(provider_uri=str(EVIDENCE_ROOT / PLAN["providers"]["price"]["provider_uri"]), region="us", kernels=1)
calendar = D.calendar(freq="day")
price_readback = D.features(["SOXL", "SOXX"], ["$open", "$close"], start_time="2026-09-21", end_time=PLAN["as_of"], freq="day")
assert not price_readback.empty
print({"provider_uri": PLAN["providers"]["price"]["provider_uri"], "region": "us", "as_of": PLAN["as_of"], "calendar_sessions": len(calendar)})
display(Markdown((CARD / "progress_review.md").read_text()))


2026/10/12 10:50:29 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[51231:MainThread](2026-10-12 10:50:31,480) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[51231:MainThread](2026-10-12 10:50:31,482) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[51231:MainThread](2026-10-12 10:50:31,482) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


{'provider_uri': 'data/qlib/published_futu_affine_20260925_v1', 'region': 'us', 'as_of': '2026-09-25', 'calendar_sessions': 4162}


# 最近进展复核（2026-10-12）

研究证据比此前更可信：时点、标签成熟、费用独立路径、Qlib原生现金/数量对账与失败封存均已有明确证据。但当前没有同时达到扣费后SOXX稳定超额与回撤门的策略。

- 层归因：冻结原始配置在开发期相对SOXX财富1.2006/1.0747，在复用诊断期降为0.8144/0.7758（10/25bp）；原始预测的泛化不足。风险投影减少回撤，也减少上涨暴露，不能靠移除风险门宣称解决。
- H20：55次拟合、8个原生场景后，组合退回SOXX；预测MSE未超越各自prior，路线关闭。
- VXN/VIX：100次拟合、12个原生场景。复用诊断门通过，开发稳定门与匹配控制组信息/组合增量未通过；SOXL未被使用，路线关闭。
- 原始季度指引：110次拟合、12个原生场景。原始文本/保守发布时点资格得到改善，超额预测和净组合增量未通过；累计正超额集中于少数年份，全部场景SOXL数量为零，路线关闭。
- 活跃SIM仍是2026-09-28 paper trial身份，开发期MaxDD/CVaR比较门失败。当前raw observed与settled日期混有旧回放，不能当作独立合格前向。独立paper有近期事件记录；完整认证资格仍是needs_data，原始计数见progress_snapshot.json。

本轮有限优化：只改变选模型的评价口径，比较平均两腿归一化MSE与H5收益差MSE。内层模型完全复用，只有选择改变的年份允许一次外层成熟样本重拟合；费用、期限、风险与候选集合固定。若失败封存，不继续重搜同一路线。研究入口是本卡notebook，预测、模型、组合明细与原生报告归Recorder。

父结论：../stage1_uniform_layer_attribution_20261010/conclusion.md、../h20_matched_forecast_execution_20261010/conclusion.md、../vxn_vix_information_increment_20261011/conclusion.md、../nvda_original_guidance_information_20261011/conclusion.md。进展来自本次文件与运行状态复核，旧聊天不作为持久状态。


In [2]:
inputs = load_frozen_selection_inputs(evidence_root=EVIDENCE_ROOT, source_root=SOURCE_ROOT, plan=PLAN)
assert isinstance(inputs["bundle"]["control"], DatasetH)
prepared = inputs["bundle"]["control"].prepare("all", col_set=["feature", "label", "metadata"])
assert "information_value" not in prepared["feature"]
print({"native_dataset_rows": len(prepared), "features": len(prepared["feature"].columns), "parent_recorder": inputs["source_audit"]["recorder_id"], "source_read_only": inputs["source_audit"]["read_only"]})


[51231:MainThread](2026-10-12 10:50:31,626) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[51231:MainThread](2026-10-12 10:50:31,626) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[51231:MainThread](2026-10-12 10:50:31,626) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[51231:MainThread](2026-10-12 10:50:31,632) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[51231:MainThread](2026-10-12 10:50:31,633) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[51231:MainThread](2026-10-12 10:50:31,633) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{'native_dataset_rows': 3956, 'features': 25, 'parent_recorder': '6ec729a2e81348d697ee6478d7360f57', 'source_read_only': True}


In [3]:
summary = run_relative_loss_experiment(evidence_root=EVIDENCE_ROOT, source_root=SOURCE_ROOT, card=CARD, plan_path=CARD / "plan.json", inputs=inputs)
print({k: summary[k] for k in ["recorder_id", "new_fit_calls", "changed_selection_years", "native_cases", "inherited_native_cases", "recomputed_native_cases", "historical_gate_pass", "incremental_information_pass", "incremental_portfolio_pass", "promotion_decision"]})


[51231:MainThread](2026-10-12 10:50:32,729) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name benchmark_relative_loss_selection_20261012.


[51231:MainThread](2026-10-12 10:50:32,732) INFO - qlib.workflow - [exp.py:258] - Experiment 191605371033296334 starts running ...


[51231:MainThread](2026-10-12 10:50:32,848) INFO - qlib.workflow - [recorder.py:345] - Recorder 55316eb768444cb69c115fae68b391c5 starts running under Experiment 191605371033296334 ...


[51231:MainThread](2026-10-12 10:50:33,754) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


2016: prior -> prior


...

native reused_diagnostic::candidate::25: inherited passed


[51231:MainThread](2026-10-12 10:50:44,014) INFO - qlib.workflow - [exp.py:258] - Experiment 191605371033296334 starts running ...


[51231:MainThread](2026-10-12 10:50:44,017) INFO - qlib.workflow - [recorder.py:345] - Recorder 55316eb768444cb69c115fae68b391c5 starts running under Experiment 191605371033296334 ...


[51231:MainThread](2026-10-12 10:50:44,549) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'recorder_id': '55316eb768444cb69c115fae68b391c5', 'new_fit_calls': 0, 'changed_selection_years': [], 'native_cases': 12, 'inherited_native_cases': 12, 'recomputed_native_cases': 0, 'historical_gate_pass': False, 'incremental_information_pass': False, 'incremental_portfolio_pass': False, 'promotion_decision': 'not_promoted'}


In [4]:
from src.core.recorder_sources import load_verified_recorder_objects
objects, readback_audit = load_verified_recorder_objects(CARD, expected_recorder_id=summary["recorder_id"], artifacts=["aggregate/tables.pkl"])
tables = objects["aggregate/tables.pkl"]
display(tables["model_selection.csv"])
display(tables["backtest_benchmark_comparison.csv"])
display(tables["forecast_information.csv"])
display(tables["position_usage.csv"])
with R.uri_context(summary["recorder_uri"]):
    assert R.get_recorder(experiment_name=summary["experiment"], recorder_id=summary["recorder_id"]).status == "FINISHED"
display(Markdown((CARD / "conclusion.md").read_text()))


,year,parent_selected,relative_selected,changed,validation_rows,selection_uses_outer_outcomes
0,2016,prior,prior,False,246,False
1,2017,ridge_1000,ridge_1000,False,246,False
2,2018,prior,prior,False,245,False
3,2019,ridge_1000,ridge_1000,False,245,False
4,2020,prior,prior,False,246,False
5,2021,ridge_1000,ridge_1000,False,247,False
6,2022,ridge_10,ridge_10,False,246,False
7,2023,prior,prior,False,245,False
8,2024,prior,prior,False,244,False
9,2025,prior,prior,False,246,False


,strategy,cost_bps,window,sessions,start_signal,last_exit,terminal_wealth,return_pct,cagr_pct,annualized_log_volatility_pct,...,relative_wealth,relative_return_pct,annualized_log_excess,annualized_relative_return_pct,relative_max_drawdown_pct,up_day_conditional_relative_return_pct,down_day_conditional_relative_return_pct,conditional_returns_are_noncontiguous_statistics,risk_policy,annualization_sessions
0,candidate,10.0,development,1762,2016-01-04,2023-01-04,4.666268,366.626799,24.645204,30.397708,...,1.096016,9.601650,0.013112,1.319867,-4.898765,-11.920357,24.434712,True,reported against benchmark; no implicit fixed ...,252
1,candidate,10.0,reused_diagnostic,934,2023-01-03,2026-09-25,5.266139,426.613878,56.553894,36.852089,...,1.057195,5.719533,0.015007,1.511971,-8.526311,-12.935833,21.427146,True,reported against benchmark; no implicit fixed ...,252
2,candidate,25.0,development,1762,2016-01-04,2023-01-04,4.734460,373.445961,24.904102,30.463653,...,1.113700,11.369978,0.015401,1.552062,-0.249377,-0.249377,11.648403,True,reported against benchmark; no implicit fixed ...,252
3,candidate,25.0,reused_diagnostic,934,2023-01-03,2026-09-25,4.973782,397.378217,54.159808,37.342767,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,True,reported against benchmark; no implicit fixed ...,252
4,control,10.0,development,1762,2016-01-04,2023-01-04,4.666268,366.626799,24.645204,30.397708,...,1.096016,9.601650,0.013112,1.319867,-4.898765,-11.920357,24.434712,True,reported against benchmark; no implicit fixed ...,252
5,control,10.0,reused_diagnostic,934,2023-01-03,2026-09-25,5.266139,426.613878,56.553894,36.852089,...,1.057195,5.719533,0.015007,1.511971,-8.526311,-12.935833,21.427146,True,reported against benchmark; no implicit fixed ...,252
6,control,25.0,development,1762,2016-01-04,2023-01-04,4.734460,373.445961,24.904102,30.463653,...,1.113700,11.369978,0.015401,1.552062,-0.249377,-0.249377,11.648403,True,reported against benchmark; no implicit fixed ...,252
7,control,25.0,reused_diagnostic,934,2023-01-03,2026-09-25,4.973782,397.378217,54.159808,37.342767,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,True,reported against benchmark; no implicit fixed ...,252
8,soxx,10.0,development,1762,2016-01-04,2023-01-04,4.257480,325.747970,23.021484,30.551980,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,True,reported against benchmark; no implicit fixed ...,252
9,soxx,10.0,reused_diagnostic,934,2023-01-03,2026-09-25,4.981235,398.123539,54.222102,37.342027,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,True,reported against benchmark; no implicit fixed ...,252


,window,response,rows,candidate_mse,matched_control_mse,own_prior_mse,mse_ratio_to_control,mse_ratio_to_prior,candidate_rank_ic,control_rank_ic,incremental_mse_pass,ic_is_descriptive_not_significance
0,development,soxl,1762,0.014881,0.014881,0.014652,1.0,1.015638,-0.004667,-0.004667,False,True
1,development,soxx,1762,0.001648,0.001648,0.001619,1.0,1.017721,-0.000954,-0.000954,False,True
2,development,excess,1762,0.006669,0.006669,0.006574,1.0,1.014389,-0.009192,-0.009192,False,True
3,reused_diagnostic,soxl,930,0.022460,0.022460,0.022270,1.0,1.008527,-0.059355,-0.059355,False,True
4,reused_diagnostic,soxx,930,0.002407,0.002407,0.002394,1.0,1.005222,-0.054891,-0.054891,False,True
5,reused_diagnostic,excess,930,0.010223,0.010223,0.010124,1.0,1.009820,-0.061437,-0.061437,False,True


,window,arm,cost_bps,sessions,soxl_positive_sessions,maximum_soxl_weight,cash_positive_sessions,mean_cash_weight
0,development,control,10.0,1762,0,1.433467e-14,35,1.986379e-02
1,development,soxx,10.0,1762,0,0.000000e+00,0,-6.472752e-17
2,development,candidate,10.0,1762,0,1.433467e-14,35,1.986379e-02
3,development,control,25.0,1762,0,1.109051e-14,5,2.837684e-03
4,development,soxx,25.0,1762,0,0.000000e+00,0,-1.979671e-17
5,development,candidate,25.0,1762,0,1.109051e-14,5,2.837684e-03
6,reused_diagnostic,control,10.0,934,0,2.100417e-14,10,1.070664e-02
7,reused_diagnostic,soxx,10.0,934,0,0.000000e+00,0,-7.623193e-17
8,reused_diagnostic,candidate,10.0,934,0,2.100417e-14,10,1.070664e-02
9,reused_diagnostic,control,25.0,934,0,1.094291e-14,0,1.425532e-14


# 超额误差选模型：冻结对照结果

复用既有四候选的内层拟合，仅改变选模型损失。0 次新外层拟合，零次新内层拟合；原生账户场景12/12通过（继承12，重算0）。
选择改变年份：[]。H5、价格特征、日风险预算、稳健半径和10/25bp费用保持配对。

- development/10bp：收益 366.63%，相对SOXX终值 1.096016，MaxDD -41.21%。
- reused_diagnostic/10bp：收益 426.61%，相对SOXX终值 1.057195，MaxDD -42.79%。
- development/25bp：收益 373.45%，相对SOXX终值 1.113700，MaxDD -41.39%。
- reused_diagnostic/25bp：收益 397.38%，相对SOXX终值 1.000000，MaxDD -42.79%。

历史稳定门=False；复用诊断门=False；超额预测MSE增量=False；对匹配控制组净组合增量=False。
晋升决定：不晋升；路线：end_relative_selector_route_no_further_search。
该实验只检验选模型的评价口径；固定alpha下线性Ridge直接拟合收益差，与两腿预测相减等价，不凭标签换名制造信息增量。
匹配样本继承父价格控制组的来源资格掩码；未使用原始指引数值。历史已反复查看，属于复用诊断，不能再称新样本外。
费用各自重跑配置与结算；继承场景读取原生数量/现金/费用证据，不宣称重新回放。SOXL与现金实际使用见position_usage.csv。
下一步：若本机制失败，封存该选择口径路线；新增机制须先提出可验证信息与目标的关系并冻结，再取得独立前向证据。
Qlib Recorder 55316eb768444cb69c115fae68b391c5；FINISHED。


# 超额误差选模型：冻结对照结果

复用既有四候选的内层拟合，仅改变选模型损失。0 次新外层拟合，零次新内层拟合；原生账户场景12/12通过（继承12，重算0）。
选择改变年份：[]。H5、价格特征、日风险预算、稳健半径和10/25bp费用保持配对。

- development/10bp：收益 366.63%，相对SOXX终值 1.096016，MaxDD -41.21%。
- reused_diagnostic/10bp：收益 426.61%，相对SOXX终值 1.057195，MaxDD -42.79%。
- development/25bp：收益 373.45%，相对SOXX终值 1.113700，MaxDD -41.39%。
- reused_diagnostic/25bp：收益 397.38%，相对SOXX终值 1.000000，MaxDD -42.79%。

历史稳定门=False；复用诊断门=False；超额预测MSE增量=False；对匹配控制组净组合增量=False。
晋升决定：不晋升；路线：end_relative_selector_route_no_further_search。
该实验只检验选模型的评价口径；固定alpha下线性Ridge直接拟合收益差，与两腿预测相减等价，不凭标签换名制造信息增量。
匹配样本继承父价格控制组的来源资格掩码；未使用原始指引数值。历史已反复查看，属于复用诊断，不能再称新样本外。
费用各自重跑配置与结算；继承场景读取原生数量/现金/费用证据，不宣称重新回放。SOXL与现金实际使用见position_usage.csv。
下一步：若本机制失败，封存该选择口径路线；新增机制须先提出可验证信息与目标的关系并冻结，再取得独立前向证据。
Qlib Recorder 55316eb768444cb69c115fae68b391c5；FINISHED。


补充诊断：两种口径11年选择完全一致。超额预测MSE比历史均值prior高1.44%（开发）/0.98%（复用诊断），点rank IC分别为-0.0092/-0.0614；IC未作显著性认证。开发期正超额年仅2/7（10bp）、1/7（25bp），滚动与区块下界门均未通过。该候选集中的选模型评价口径不是主瓶颈；下一机制应先验证信息增量。

独立Qlib读回：12个原生账户场景全部通过，0份拟合的训练哈希/先验均值核验；全部场景实际SOXL数量为0，仍为SOXX/现金时机比较。来源position_usage_native.csv。累计跑赢不能替代年度、滚动与区块区间门；不晋升。
